<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase17_ewma_garch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox
# Si Colab no tiene arch: !pip install arch

# 1. Ejemplos a mano
lam = 0.94
var_n = lam * 0.25 + (1 - lam) * 1.0**2
print("EWMA, varianza y volatilidad:", round(var_n, 4), round(var_n**0.5, 4))
print("Vida media de EWMA (días):", round(np.log(0.5) / np.log(lam), 1))

omega, alpha, beta = 0.0072, 0.08, 0.90
V = omega / (1 - alpha - beta)
v1 = omega + alpha * 1.5**2 + beta * 0.36
v2 = omega + alpha * 0.2**2 + beta * v1
print("GARCH, volatilidad de largo plazo:", round(V**0.5, 3))
print("GARCH, tras susto de 1.5% y tras calma de 0.2%:", round(v1**0.5, 3), round(v2**0.5, 3))
print("GARCH, pronóstico a 1, 10, 30 y 100 días:",
      [round((V + (alpha + beta)**(h - 1) * (v1 - V))**0.5, 3) for h in (1, 10, 30, 100)])
print("Vida media del choque GARCH (días):", round(np.log(0.5) / np.log(alpha + beta), 1))

omega_g, alpha_g, gamma_g, beta_g = 0.0072, 0.05, 0.06, 0.90
base = omega_g + beta_g * 0.36
print("GJR, volatilidad tras +1.5% y tras menos 1.5%:",
      round((base + alpha_g * 1.5**2)**0.5, 3),
      round((base + (alpha_g + gamma_g) * 1.5**2)**0.5, 3))

# 2. Datos reales: USD/MXN
precio = yf.download("MXN=X", start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
rend100 = precio.pct_change().dropna() * 100   # en porcentaje, lo que espera arch

# 3. Estimaciones de volatilidad usando solo información hasta ayer
vol_ewma = np.sqrt((rend100**2).ewm(alpha=0.06, adjust=False).mean()).shift(1)
vol_30 = rend100.rolling(30).std().shift(1)

# 4. GARCH(1,1) con t de Student, estimado solo en el primer 70%
corte = int(len(rend100) * 0.7)
am = arch_model(rend100, mean="Constant", vol="GARCH", p=1, q=1, dist="t")
train = am.fit(last_obs=corte, disp="off")
print(train.params.round(4))
a, b = float(train.params["alpha[1]"]), float(train.params["beta[1]"])
print("Persistencia alfa + beta:", round(a + b, 3))
if a + b < 1:
    print("Volatilidad diaria de largo plazo:",
          round((float(train.params["omega"]) / (1 - a - b))**0.5, 3))
vol_garch = am.fix(train.params).conditional_volatility  # parámetros fijos del 70%

# 5. Evaluación en el 30% más reciente
prueba = rend100.index[corte:]
tabla = pd.DataFrame({"EWMA": vol_ewma, "Ventana 30": vol_30, "GARCH": vol_garch}).loc[prueba]
r = rend100.loc[prueba]
resumen = pd.DataFrame({
    "QLIKE (menor es mejor)": [float((np.log(tabla[c]**2) + r**2 / tabla[c]**2).mean()) for c in tabla.columns],
    "Correlación con r al cuadrado": [float(np.corrcoef(tabla[c]**2, r**2)[0, 1]) for c in tabla.columns],
    "Desv. estándar de r entre vol. prevista": [float((r / tabla[c]).std()) for c in tabla.columns],
}, index=tabla.columns)
print(resumen.round(3))

# 6. ¿El modelo captura las rachas de volatilidad?
mu = float(train.params["mu"])
z = (rend100 - mu) / vol_garch
print("Ljung Box (rezago 10) de r al cuadrado, p-valor:",
      round(float(acorr_ljungbox(rend100**2, lags=[10])["lb_pvalue"].iloc[0]), 4))
print("Ljung Box (rezago 10) de z al cuadrado, p-valor:",
      round(float(acorr_ljungbox(z**2, lags=[10])["lb_pvalue"].iloc[0]), 4))

# 7. Asimetría con GJR
gjr = arch_model(rend100, mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="t").fit(disp="off")
print("GJR, gamma y su p-valor:", round(float(gjr.params["gamma[1]"]), 3),
      round(float(gjr.pvalues["gamma[1]"]), 4))

# 8. Pronóstico a 30 días desde el último dato
completo = am.fit(disp="off")
pron = completo.forecast(horizon=30, reindex=False)
print("Volatilidad diaria pronosticada a 1, 10 y 30 días:",
      np.sqrt(pron.variance.values[-1, [0, 9, 29]]).round(3))

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox
# Si Colab no tiene arch: !pip install arch

# 1. Ejemplos a mano
lam = 0.94
var_n = lam * 0.25 + (1 - lam) * 1.0**2
print("EWMA, varianza y volatilidad:", round(var_n, 4), round(var_n**0.5, 4))
print("Vida media de EWMA (días):", round(np.log(0.5) / np.log(lam), 1))

omega, alpha, beta = 0.0072, 0.08, 0.90
V = omega / (1 - alpha - beta)
v1 = omega + alpha * 1.5**2 + beta * 0.36
v2 = omega + alpha * 0.2**2 + beta * v1
print("GARCH, volatilidad de largo plazo:", round(V**0.5, 3))
print("GARCH, tras susto de 1.5% y tras calma de 0.2%:", round(v1**0.5, 3), round(v2**0.5, 3))
print("GARCH, pronóstico a 1, 10, 30 y 100 días:",
      [round((V + (alpha + beta)**(h - 1) * (v1 - V))**0.5, 3) for h in (1, 10, 30, 100)])
print("Vida media del choque GARCH (días):", round(np.log(0.5) / np.log(alpha + beta), 1))

omega_g, alpha_g, gamma_g, beta_g = 0.0072, 0.05, 0.06, 0.90
base = omega_g + beta_g * 0.36
print("GJR, volatilidad tras +1.5% y tras menos 1.5%:",
      round((base + alpha_g * 1.5**2)**0.5, 3),
      round((base + (alpha_g + gamma_g) * 1.5**2)**0.5, 3))

# 2. Datos reales: USD/MXN
precio = yf.download("MXN=X", start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()
rend100 = precio.pct_change().dropna() * 100   # en porcentaje, lo que espera arch

# 3. Estimaciones de volatilidad usando solo información hasta ayer
vol_ewma = np.sqrt((rend100**2).ewm(alpha=0.06, adjust=False).mean()).shift(1)
vol_30 = rend100.rolling(30).std().shift(1)

# 4. GARCH(1,1) con t de Student, estimado solo en el primer 70%
corte = int(len(rend100) * 0.7)
am = arch_model(rend100, mean="Constant", vol="GARCH", p=1, q=1, dist="t")
train = am.fit(last_obs=corte, disp="off")
print(train.params.round(4))
a, b = float(train.params["alpha[1]"]), float(train.params["beta[1]"])
print("Persistencia alfa + beta:", round(a + b, 3))
if a + b < 1:
    print("Volatilidad diaria de largo plazo:",
          round((float(train.params["omega"]) / (1 - a - b))**0.5, 3))
vol_garch = am.fix(train.params).conditional_volatility  # parámetros fijos del 70%

# 5. Evaluación en el 30% más reciente
prueba = rend100.index[corte:]
tabla = pd.DataFrame({"EWMA": vol_ewma, "Ventana 30": vol_30, "GARCH": vol_garch}).loc[prueba]
r = rend100.loc[prueba]
resumen = pd.DataFrame({
    "QLIKE (menor es mejor)": [float((np.log(tabla[c]**2) + r**2 / tabla[c]**2).mean()) for c in tabla.columns],
    "Correlación con r al cuadrado": [float(np.corrcoef(tabla[c]**2, r**2)[0, 1]) for c in tabla.columns],
    "Desv. estándar de r entre vol. prevista": [float((r / tabla[c]).std()) for c in tabla.columns],
}, index=tabla.columns)
print(resumen.round(3))

# 6. ¿El modelo captura las rachas de volatilidad?
mu = float(train.params["mu"])
z = (rend100 - mu) / vol_garch
print("Ljung Box (rezago 10) de r al cuadrado, p-valor:",
      round(float(acorr_ljungbox(rend100**2, lags=[10])["lb_pvalue"].iloc[0]), 4))
print("Ljung Box (rezago 10) de z al cuadrado, p-valor:",
      round(float(acorr_ljungbox(z**2, lags=[10])["lb_pvalue"].iloc[0]), 4))

# 7. Asimetría con GJR
gjr = arch_model(rend100, mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="t").fit(disp="off")
print("GJR, gamma y su p-valor:", round(float(gjr.params["gamma[1]"]), 3),
      round(float(gjr.pvalues["gamma[1]"]), 4))

# 8. Pronóstico a 30 días desde el último dato
completo = am.fit(disp="off")
pron = completo.forecast(horizon=30, reindex=False)
print("Volatilidad diaria pronosticada a 1, 10 y 30 días:",
      np.sqrt(pron.variance.values[-1, [0, 9, 29]]).round(3))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 12.1 MB/s  0:00:00
EWMA, varianza y volatilidad: 0.295 0.5431
Vida media de EWMA (días): 11.2
GARCH, volatilidad de largo plazo: 0.6
GARCH, tras susto de 1.5% y tras calma de 0.2%: 0.715 0.686
GARCH, pronóstico a 1, 10, 30 y 100 días: [0.715, 0.697, 0.666, 0.617]
Vida media del choque GARCH (días): 34.3
GJR, volatilidad tras +1.5% y tras menos 1.5%: 0.666 0.761


/tmp/ipykernel_293/809165877.py:33: FutureWarning: YF.download() has changed argument auto_adjust default to True
  precio = yf.download("MXN=X", start="2020-10-07", end="2026-10-22", progress=False)["Close"].squeeze()


mu         -0.0588
omega       0.0616
alpha[1]    0.1875
beta[1]     0.7002
nu          6.2865
Name: params, dtype: float64
Persistencia alfa + beta: 0.888
Volatilidad diaria de largo plazo: 0.741
            QLIKE (menor es mejor)  Correlación con r al cuadrado  \
EWMA                        -0.159                          0.204   
Ventana 30                  -0.069                          0.149   
GARCH                       -0.176                          0.267   

            Desv. estándar de r entre vol. prevista  
EWMA                                          1.054  
Ventana 30                                    1.121  
GARCH                                         0.889  
Ljung Box (rezago 10) de r al cuadrado, p-valor: 0.0
Ljung Box (rezago 10) de z al cuadrado, p-valor: 0.5545
GJR, gamma y su p-valor: -0.246 0.0
Volatilidad diaria pronosticada a 1, 10 y 30 días: [0.556 0.65  0.724]


In [3]:
print(rend100.abs().sort_values(ascending=False).head(10))

Date
2024-06-04    4.288258
2025-02-04    3.704731
2025-04-07    3.459172
2023-03-14    3.286836
2024-08-05    3.195085
2021-02-26    2.617293
2024-06-10    2.493800
2024-08-09    2.453823
2025-04-10    2.412581
2025-02-03    2.393911
Name: MXN=X, dtype: float64


In [4]:
print(gjr.params.round(4))

mu         -0.0389
omega       0.0451
alpha[1]    0.2697
gamma[1]   -0.2458
beta[1]     0.7480
nu          5.8184
Name: params, dtype: float64


In [5]:
gjr_m = arch_model(rend100, mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="t")
gjr_tr = gjr_m.fit(last_obs=corte, disp="off")
vol_gjr = gjr_m.fix(gjr_tr.params).conditional_volatility.loc[prueba]

qlike = lambda v: float((np.log(v**2) + r**2 / v**2).mean())
print("QLIKE fuera de muestra, GARCH:", round(qlike(tabla["GARCH"]), 3))
print("QLIKE fuera de muestra, GJR:  ", round(qlike(vol_gjr), 3))

QLIKE fuera de muestra, GARCH: -0.176
QLIKE fuera de muestra, GJR:   -0.17
